# S24 변동 폭과 생성 평가 질의 규모

기준 설정(EXP-017 검색, 프롬프트 v2, judge j_v8)을 dev-full 판정용 순서의 앞 100질의로 반복 실행해 생성 잡음과 judge 잡음을 나누고, 판정용 질의 수 n과 judge 실행 설정을 정할 근거를 낸다(`docs/slices/24-variance.md`). 생성은 3회(반복 0은 차례로, 1·2는 동시 4), judge는 `deepseek_go_mixed`로 생성 반복마다 채점하고 생성 반복 0에 3회, `deepseek_go_nothink`로 생성 반복 0에 3회 채점했다.

- 질의 단위 값: 충실성, 답변 관련성, 핵심 답변 포함도, 거부 정확도(참고값, 질의마다 맞는 처리 / 판정 대상 항목)
- 구간은 판정 코드와 같은 회의 단위 paired bootstrap(`rag.eval.bootstrap.paired_bootstrap`, `conference_number`)
- 질의 원문은 출력에 남기지 않는다(D-31)

In [1]:
import json
from collections import Counter
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd

from rag.eval.bootstrap import paired_bootstrap
from rag.eval.metrics import load_gold
from rag.index import load_cfg

In [2]:
ROOT = Path.cwd().parent
RUNS = ROOT / "data/gen_eval/runs"
JUDGE = ROOT / "data/gen_eval/judge"
STEM = "hybrid-cond-filter-law-ms256-k20-ce-v2-dev-full"
S14_RUNS = {"S14 v1": RUNS / "hybrid-cond-filter-law-ms256-k20-ce-v1-dev-small.jsonl",
            "S14 v2": RUNS / "hybrid-cond-filter-law-ms256-k20-ce-v2-dev-small.jsonl"}
GEN_REPS = [0, 1, 2]
JUDGE_REPS = [0, 1, 2]
SETTINGS = ["deepseek_go_mixed", "deepseek_go_nothink"]
CRITERIA = ["충실성", "답변 관련성", "핵심 답변 포함도", "거부 정확도(참고)"]
N_GRID = [100, 200, 300, 500, 800, 1000]
DELTAS = [0.02, 0.03, 0.05]
# opencode Go DeepSeek V4.1 Flash 100만 토큰당(2026-10-21 가격표, D-44). 피크는 두 배
JUDGE_PRICE = {"input": 0.15, "output": 0.60}

In [3]:
def tag(n: int) -> str:
    return f"-r{n}" if n else ""


def read(path: Path) -> list[dict]:
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def judge_path(setting: str, gen_rep: int, judge_rep: int) -> Path:
    return JUDGE / f"{STEM}{tag(gen_rep)}-{setting}-j_v8{tag(judge_rep)}.jsonl"


def scores(rows: list[dict]) -> pd.DataFrame:
    """질의 단위 값. 계산하지 않은 기준(전체 거부, 참조 사용 불가, 형식 오류)은 NaN."""
    out = {}
    for r in rows:
        counts = (r.get("item_eval") or {}).get("counts")
        acc = counts["correct"] / counts["judged"] if counts and counts["judged"] else np.nan
        out[r["qid"]] = {"충실성": (r.get("faithfulness") or {}).get("score"),
                         "답변 관련성": (r.get("relevance") or {}).get("score"),
                         "핵심 답변 포함도": (r.get("inclusion") or {}).get("score"),
                         "거부 정확도(참고)": acc}
    return pd.DataFrame.from_dict(out, orient="index").astype(float)


def calls(rows: list[dict]) -> list[dict]:
    keys = ("faithfulness", "relevance", "inclusion", "key_statements", "chunk_keys", "item_eval")
    return [u for r in rows for k in keys for u in ((r.get(k) or {}).get("usage") or []) if u]


def half_width(a: pd.Series, b: pd.Series, groups: pd.Series, boot) -> tuple[float, int, float]:
    """두 실행의 질의 단위 값 차이에 대한 95% 구간 반폭, 공통 질의 수, 한쪽만 계산된 비율."""
    both = a.notna() & b.notna()
    one = a.notna() ^ b.notna()
    _, lo, hi = paired_bootstrap((a - b)[both], groups[both[both].index], boot.n_resamples,
                                 boot.seed, boot.alpha)
    return (hi - lo) / 2, int(both.sum()), float(one.sum() / (both.sum() + one.sum()))

## 데이터 로딩

In [4]:
cfg = load_cfg(["index.scope=full", "+exp=exp017"])
boot = cfg.eval.bootstrap
gens = {g: read(RUNS / f"{STEM}{tag(g)}.jsonl") for g in GEN_REPS}
runs = {}
for s in SETTINGS:
    for g in GEN_REPS:
        for j in JUDGE_REPS:
            p = judge_path(s, g, j)
            if p.exists():
                runs[s, g, j] = read(p)
groups = load_gold(cfg, "dev-full")["conference_number"]
print({g: len(v) for g, v in gens.items()})
print({k: len(v) for k, v in runs.items()})
print("회의 수", groups[[r["qid"] for r in gens[0]]].nunique())
print("생성 기록 qid가 반복마다 같은가", all([r["qid"] for r in gens[g]] == [r["qid"] for r in gens[0]] for g in GEN_REPS))

{0: 100, 1: 100, 2: 100}
{('deepseek_go_mixed', 0, 0): 100, ('deepseek_go_mixed', 0, 1): 100, ('deepseek_go_mixed', 0, 2): 100, ('deepseek_go_mixed', 1, 0): 100, ('deepseek_go_mixed', 2, 0): 100, ('deepseek_go_nothink', 0, 0): 100, ('deepseek_go_nothink', 0, 1): 100, ('deepseek_go_nothink', 0, 2): 100}
회의 수 75
생성 기록 qid가 반복마다 같은가 True


In [5]:
# 검색 근거가 반복·동시 수와 관계없이 같은지(완료 기준 3)
ctx = {g: {r["qid"]: [c["chunk_id"] for c in r["contexts"]] for r in gens[g]} for g in GEN_REPS}
print("반복 0(차례로)과 같은 근거:", {g: sum(ctx[g][q] == ctx[0][q] for q in ctx[0]) for g in GEN_REPS})
print("동시 수:", {g: Counter(r["concurrency"] for r in gens[g]) for g in GEN_REPS})
tables = {k: scores(v) for k, v in runs.items()}
for k, t in tables.items():
    print(k, "평균", t.mean().round(3).to_dict(), "계산한 질의", t.notna().sum().to_dict())

반복 0(차례로)과 같은 근거: {0: 100, 1: 100, 2: 100}
동시 수: {0: Counter({1: 100}), 1: Counter({4: 100}), 2: Counter({4: 100})}
('deepseek_go_mixed', 0, 0) 평균 {'충실성': 0.765, '답변 관련성': 0.904, '핵심 답변 포함도': 0.608, '거부 정확도(참고)': 0.618} 계산한 질의 {'충실성': 92, '답변 관련성': 94, '핵심 답변 포함도': 73, '거부 정확도(참고)': 96}
('deepseek_go_mixed', 0, 1) 평균 {'충실성': 0.778, '답변 관련성': 0.92, '핵심 답변 포함도': 0.575, '거부 정확도(참고)': 0.548} 계산한 질의 {'충실성': 93, '답변 관련성': 94, '핵심 답변 포함도': 75, '거부 정확도(참고)': 94}
('deepseek_go_mixed', 0, 2) 평균 {'충실성': 0.778, '답변 관련성': 0.926, '핵심 답변 포함도': 0.604, '거부 정확도(참고)': 0.596} 계산한 질의 {'충실성': 93, '답변 관련성': 94, '핵심 답변 포함도': 76, '거부 정확도(참고)': 90}
('deepseek_go_mixed', 1, 0) 평균 {'충실성': 0.783, '답변 관련성': 0.917, '핵심 답변 포함도': 0.568, '거부 정확도(참고)': 0.597} 계산한 질의 {'충실성': 95, '답변 관련성': 96, '핵심 답변 포함도': 76, '거부 정확도(참고)': 96}
('deepseek_go_mixed', 2, 0) 평균 {'충실성': 0.777, '답변 관련성': 0.887, '핵심 답변 포함도': 0.586, '거부 정확도(참고)': 0.628} 계산한 질의 {'충실성': 95, '답변 관련성': 97, '핵심 답변 포함도': 77, '거부 정확도(참고)': 96}
('deepseek_go_nothink', 0

## 잡음 분해

같은 답변을 judge만 다시 채점한 차이는 judge 잡음, 생성부터 다시 한 차이는 생성 잡음과 judge 잡음의 합이다. 질의마다 반복 사이 표준편차를 내고 질의 평균을 본다. 생성 잡음은 두 분산의 차이로 어림한다(음수면 0).

온도 0이라 생성 반복에서 답변이 글자까지 같은 질의가 있고, 같은 답변은 judge 요청도 같아 저장 응답을 쓴다(judge 반복 0끼리). 그래서 "생성부터 다시, judge 반복 같음"의 차이에는 답변이 같은 질의의 judge 잡음이 빠진다. 설정 비교에서는 답변이 달라 judge를 새로 부르므로, n은 생성 반복과 judge 반복이 둘 다 다른 짝("생성·judge 모두 다시")으로 정한다.

In [6]:
# 생성 반복 사이 답변이 글자까지 같은 질의 수
ans = {g: {r["qid"]: r["answer"] for r in gens[g]} for g in GEN_REPS}
print({f"{a}-{b}": sum(ans[a][q] == ans[b][q] for q in ans[0]) for a, b in combinations(GEN_REPS, 2)}, "/", len(ans[0]))

{'0-1': 20, '0-2': 19, '1-2': 16} / 100


In [7]:
def rep_sd(keys: list[tuple]) -> pd.Series:
    stack = np.stack([tables[k][CRITERIA].reindex(tables[keys[0]].index).to_numpy() for k in keys])
    return pd.Series(np.nanmean(np.nanstd(stack, axis=0, ddof=1), axis=0), index=CRITERIA)


judge_sd = rep_sd([("deepseek_go_mixed", 0, j) for j in JUDGE_REPS])
total_sd = rep_sd([("deepseek_go_mixed", g, 0) for g in GEN_REPS])
noise = pd.DataFrame({"judge 잡음 SD": judge_sd, "생성+judge SD": total_sd,
                      "생성 잡음 SD(어림)": np.sqrt(np.clip(total_sd**2 - judge_sd**2, 0, None))})
noise.round(3)

C:\workspace\meet-rag\.venv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1997: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


,judge 잡음 SD,생성+judge SD,생성 잡음 SD(어림)
충실성,0.073,0.101,0.070
답변 관련성,0.031,0.058,0.049
핵심 답변 포함도,0.060,0.058,0.000
거부 정확도(참고),0.107,0.030,0.000


In [8]:
# 같은 설정 두 실행의 평균 차이 95% 구간 반폭 h(n0)와 한쪽만 계산된 비율(D-47 결정 6)
rows = []
pairs = {"judge만 다시": [(("deepseek_go_mixed", 0, a), ("deepseek_go_mixed", 0, b)) for a, b in combinations(JUDGE_REPS, 2)],
         "생성부터 다시, judge 반복 같음": [(("deepseek_go_mixed", a, 0), ("deepseek_go_mixed", b, 0)) for a, b in combinations(GEN_REPS, 2)],
         "생성·judge 모두 다시": [(("deepseek_go_mixed", 0, j), ("deepseek_go_mixed", g, 0)) for j in (1, 2) for g in (1, 2)]}
for kind, ps in pairs.items():
    for c in CRITERIA:
        hs = [half_width(tables[x][c], tables[y][c], groups, boot) for x, y in ps]
        rows.append({"반복": kind, "기준": c, "h(n0)": np.mean([h for h, _, _ in hs]),
                     "공통 질의": np.mean([n for _, n, _ in hs]), "한쪽만 계산 비율": np.mean([e for _, _, e in hs])})
hw = pd.DataFrame(rows)
hw.round(3)

,반복,기준,h(n0),공통 질의,한쪽만 계산 비율
0,judge만 다시,충실성,0.046,92.333,0.007
1,judge만 다시,답변 관련성,0.030,94.000,0.000
2,judge만 다시,핵심 답변 포함도,0.039,72.000,0.069
3,judge만 다시,거부 정확도(참고),0.063,89.333,0.082
4,"생성부터 다시, judge 반복 같음",충실성,0.057,92.000,0.042
5,"생성부터 다시, judge 반복 같음",답변 관련성,0.039,94.667,0.021
6,"생성부터 다시, judge 반복 같음",핵심 답변 포함도,0.042,74.000,0.035
7,"생성부터 다시, judge 반복 같음",거부 정확도(참고),0.031,96.000,0.000
8,생성·judge 모두 다시,충실성,0.055,91.500,0.052
9,생성·judge 모두 다시,답변 관련성,0.036,94.000,0.026


In [9]:
# n에 따른 반폭 h(n) ≈ h(n0)·√(n0/n)과 δ별 필요한 n(한쪽만 계산 비율로 나눠 키움). 생성·judge 모두 다시 기준
base = hw[hw["반복"] == "생성·judge 모두 다시"].set_index("기준")
table = {}
for c in CRITERIA:
    h0, n0, ex = base.loc[c, "h(n0)"], base.loc[c, "공통 질의"], base.loc[c, "한쪽만 계산 비율"]
    row = {f"h({n})": h0 * np.sqrt(n0 / n) for n in N_GRID}
    row |= {f"δ={d} 필요 n": int(np.ceil(n0 * (h0 / d) ** 2 / (1 - ex))) for d in DELTAS}
    table[c] = row
pd.DataFrame(table).T.round(3)

,h(100),h(200),h(300),h(500),h(800),h(1000),δ=0.02 필요 n,δ=0.03 필요 n,δ=0.05 필요 n
충실성,0.053,0.037,0.030,0.024,0.019,0.017,729.0,324.0,117.0
답변 관련성,0.035,0.025,0.020,0.016,0.012,0.011,310.0,138.0,50.0
핵심 답변 포함도,0.047,0.033,0.027,0.021,0.017,0.015,603.0,268.0,97.0
거부 정확도(참고),0.065,0.046,0.038,0.029,0.023,0.021,1154.0,513.0,185.0


## 실행 설정 비교

같은 답변(생성 반복 0)을 설정마다 3회 채점한 일관성이다. 진술·항목은 반복마다 다시 뽑혀 판정 단위로 맞출 수 없어 질의 단위 값으로 본다. 다수결(3회 중앙값)은 반복 3회 기록에서 계산할 수 있지만, 그 자체의 일관성은 독립 다수결 두 개가 필요해 여기서는 재지 못한다.

In [10]:
rows = []
for s in SETTINGS:
    ks = [(s, 0, j) for j in JUDGE_REPS if (s, 0, j) in tables]
    for c in CRITERIA:
        diffs, same = [], []
        for x, y in combinations(ks, 2):
            a, b = tables[x][c], tables[y][c]
            both = a.notna() & b.notna()
            diffs.append((a - b)[both].abs().mean())
            same.append(((a - b)[both].abs() < 1e-9).mean())
        rows.append({"설정": s, "기준": c, "반복 사이 |차| 평균": np.mean(diffs), "같은 값 비율": np.mean(same),
                     "평균(반복 평균)": np.mean([tables[k][c].mean() for k in ks])})
pd.DataFrame(rows).round(3)

,설정,기준,반복 사이 |차| 평균,같은 값 비율,평균(반복 평균)
0,deepseek_go_mixed,충실성,0.087,0.809,0.774
1,deepseek_go_mixed,답변 관련성,0.035,0.936,0.917
2,deepseek_go_mixed,핵심 답변 포함도,0.068,0.764,0.596
3,deepseek_go_mixed,거부 정확도(참고),0.122,0.813,0.587
4,deepseek_go_nothink,충실성,0.010,0.978,0.975
5,deepseek_go_nothink,답변 관련성,0.035,0.936,0.917
6,deepseek_go_nothink,핵심 답변 포함도,0.029,0.916,0.564
7,deepseek_go_nothink,거부 정확도(참고),0.052,0.923,0.488


In [14]:
# 일관성이 높은 이유: 같은 답변·같은 진술에서 verdicts의 0(뒷받침 안 됨) 판정 수
for s in SETTINGS:
    for j in JUDGE_REPS:
        v = [x["verdict"] for r in runs[s, 0, j] for x in ((r.get("faithfulness") or {}).get("verdicts") or [])]
        print(s, f"judge 반복 {j}", "진술", len(v), "0 판정", v.count(0), f"({v.count(0) / len(v):.1%})")
same = sum(1 for x, y in zip(runs["deepseek_go_mixed", 0, 0], runs["deepseek_go_nothink", 0, 0], strict=True)
           if [v["statement"] for v in (x.get("faithfulness") or {}).get("verdicts") or []]
           == [v["statement"] for v in (y.get("faithfulness") or {}).get("verdicts") or []])
print("두 설정의 진술 목록이 같은 질의(judge 반복 0)", same, "/ 100")

deepseek_go_mixed judge 반복 0 진술 236 0 판정 53 (22.5%)
deepseek_go_mixed judge 반복 1 진술 239 0 판정 49 (20.5%)
deepseek_go_mixed judge 반복 2 진술 237 0 판정 53 (22.4%)
deepseek_go_nothink judge 반복 0 진술 233 0 판정 6 (2.6%)
deepseek_go_nothink judge 반복 1 진술 231 0 판정 6 (2.6%)
deepseek_go_nothink judge 반복 2 진술 237 0 판정 6 (2.5%)
두 설정의 진술 목록이 같은 질의(judge 반복 0) 98 / 100


In [11]:
# 단계별 새 호출의 지연·토큰·끝나지 않은 응답과 비용 어림(비피크, 피크는 두 배)
rows = []
for s in SETTINGS:
    us = [u for (ss, g, j), v in runs.items() if ss == s for u in calls(v) if not u["cached"]]
    tin = sum(u.get("prompt_tokens") or 0 for u in us)
    tout = sum(u.get("completion_tokens") or 0 for u in us)
    lat = [u["latency_s"] for u in us if u.get("latency_s") is not None]
    rows.append({"설정": s, "새 호출": len(us), "끝나지 않음": sum(u.get("finish_reason") != "stop" for u in us),
                 "입력 토큰": tin, "출력 토큰": tout,
                 "비피크 비용 $": round((tin * JUDGE_PRICE["input"] + tout * JUDGE_PRICE["output"]) / 1e6, 2),
                 "지연 중앙값 s": round(float(np.median(lat)), 1), "지연 p95 s": round(float(np.percentile(lat, 95)), 1)})
pd.DataFrame(rows)

,설정,새 호출,끝나지 않음,입력 토큰,출력 토큰,비피크 비용 $,지연 중앙값 s,지연 p95 s
0,deepseek_go_mixed,2943,1,4076639,2371479,2.03,2.2,34.2
1,deepseek_go_nothink,1579,0,2683106,264974,0.56,2.4,3.8


In [12]:
# 단계별 지연(새 호출만)
rows = []
for s in SETTINGS:
    us = [u for (ss, g, j), v in runs.items() if ss == s for u in calls(v) if not u["cached"]]
    for step in dict.fromkeys(u["step"] for u in us):
        lat = [u["latency_s"] for u in us if u["step"] == step]
        out = [u.get("completion_tokens") or 0 for u in us if u["step"] == step]
        rows.append({"설정": s, "단계": step, "호출": len(lat), "지연 중앙값 s": np.median(lat),
                     "지연 p95 s": np.percentile(lat, 95), "출력 토큰 중앙값": np.median(out)})
pd.DataFrame(rows).round(1)

,설정,단계,호출,지연 중앙값 s,지연 p95 s,출력 토큰 중앙값
0,deepseek_go_mixed,statements,437,1.8,2.6,97.0
1,deepseek_go_mixed,verdicts,430,15.3,57.4,1867.0
2,deepseek_go_mixed,relevance,438,1.6,2.3,68.0
3,deepseek_go_mixed,key_checks,345,2.2,3.6,183.0
4,deepseek_go_mixed,key_statements,294,4.1,10.4,357.5
5,deepseek_go_mixed,chunk_checks,244,2.3,3.9,188.0
6,deepseek_go_mixed,items,300,23.4,59.6,2758.0
7,deepseek_go_mixed,item_handling,455,1.6,2.4,71.0
8,deepseek_go_nothink,verdicts,279,2.8,4.3,255.0
9,deepseek_go_nothink,key_checks,224,2.4,3.7,178.5


In [15]:
# 질의 안의 독립 사슬을 동시에 부를 때의 질의당 시간 어림(사용자 지시, 측정 뒤 구현).
# 모든 호출을 새로 부른 실행(mixed judge 반복 1·2, nothink 반복 1·2)의 호출 지연으로 계산한다.
# 차례로: 지연 합. 동시: 사슬별 지연 합 중 최댓값(충실성 statements→verdicts, 관련성, 포함도
# key_statements→chunk_checks→key_checks, 항목 items→item_handling)
CHAINS = {"충실성": ("statements", "verdicts"), "관련성": ("relevance",),
          "포함도": ("key_statements", "chunk_checks", "key_checks"), "항목": ("items", "item_handling")}
rows = []
for s in SETTINGS:
    seq, par = [], []
    for j in (1, 2):
        for r in runs[s, 0, j]:
            lat = Counter()
            for u in calls([r]):
                lat[u["step"]] += u.get("latency_s") or 0
            seq.append(sum(lat.values()))
            par.append(max(sum(lat[st] for st in steps) for steps in CHAINS.values()))
    rows.append({"설정": s, "질의": len(seq), "차례로 질의당 중앙값 s": np.median(seq),
                 "동시 질의당 중앙값 s": np.median(par),
                 "100질의 예상 분(질의 동시 4, 차례로)": sum(seq) / 2 / 4 / 60,
                 "100질의 예상 분(질의 동시 4, 사슬 동시)": sum(par) / 2 / 4 / 60})
pd.DataFrame(rows).round(1)

,설정,질의,차례로 질의당 중앙값 s,동시 질의당 중앙값 s,"100질의 예상 분(질의 동시 4, 차례로)","100질의 예상 분(질의 동시 4, 사슬 동시)"
0,deepseek_go_mixed,200,52.1,28.4,24.5,13.5
1,deepseek_go_nothink,200,17.4,6.4,7.1,2.8


In [16]:
# verdicts만 low인 설정(deepseek_go_verdicts, 사용자 제안). 요청 본문이 단계마다 mixed(verdicts)나
# nothink(나머지)와 같아 저장 응답으로 채점했다(API 호출 0회). 지연은 저장 응답의 원래 지연이다
for j in JUDGE_REPS:
    runs["deepseek_go_verdicts", 0, j] = read(judge_path("deepseek_go_verdicts", 0, j))
    tables["deepseek_go_verdicts", 0, j] = scores(runs["deepseek_go_verdicts", 0, j])
rows = []
for s in SETTINGS + ["deepseek_go_verdicts"]:
    ks = [(s, 0, j) for j in JUDGE_REPS]
    v = [x["verdict"] for k in ks for r in runs[k] for x in ((r.get("faithfulness") or {}).get("verdicts") or [])]
    row = {"설정": s, "verdicts 0 판정": f"{v.count(0) / len(v):.1%}"}
    for c in CRITERIA:
        diffs = []
        for x, y in combinations(ks, 2):
            a, b = tables[x][c], tables[y][c]
            both = a.notna() & b.notna()
            diffs.append((a - b)[both].abs().mean())
        row[f"{c} 평균"] = np.mean([tables[k][c].mean() for k in ks])
        row[f"{c} |차|"] = np.mean(diffs)
    seq, par = [], []
    for j in (1, 2):
        for r in runs[s, 0, j]:
            lat = Counter()
            for u in calls([r]):
                lat[u["step"]] += u.get("latency_s") or 0
            seq.append(sum(lat.values()))
            par.append(max(sum(lat[st] for st in steps) for steps in CHAINS.values()))
    row["100질의 분(사슬 동시)"] = sum(par) / 2 / 4 / 60
    rows.append(row)
pd.DataFrame(rows).set_index("설정").T

설정,deepseek_go_mixed,deepseek_go_nothink,deepseek_go_verdicts
verdicts 0 판정,21.8%,2.6%,21.8%
충실성 평균,0.773762,0.975332,0.773762
충실성 |차|,0.086814,0.009695,0.086814
답변 관련성 평균,0.916667,0.916667,0.916667
답변 관련성 |차|,0.035461,0.035461,0.035461
핵심 답변 포함도 평균,0.595839,0.564135,0.564135
핵심 답변 포함도 |차|,0.068018,0.028973,0.028973
거부 정확도(참고) 평균,0.587346,0.487884,0.487884
거부 정확도(참고) |차|,0.122456,0.052323,0.052323
100질의 분(사슬 동시),13.514089,2.804387,8.158127


In [17]:
# 시간 단축 검토(사용자 요청): mixed, n=330 기준 judge 시간 어림. 같은 호출 지연을 가정한다.
# 답변과 무관한 단계(key_statements, chunk_checks, items)는 검색 근거가 같으면 기준 설정의 저장 응답을 쓴다.
# 생성 쪽만 바꾸는 실험(프롬프트 등)은 이 세 단계가 0초다. 실제 측정: 생성 반복 1·2 채점은 답변 무관 단계가
# 저장 응답이라 사슬을 차례로 부르고도 100질의에 약 9분이었다(13:59~14:08, 14:08~14:17)
ANSWER_FREE = {"key_statements", "chunk_checks", "items"}
N = 330
rows = []
for name, drop in [("검색 바뀜(모든 단계 새로)", set()), ("생성만 바뀜(답변 무관 단계 저장 응답)", ANSWER_FREE)]:
    seq, par = [], []
    for j in (1, 2):
        for r in runs["deepseek_go_mixed", 0, j]:
            lat = Counter()
            for u in calls([r]):
                if u["step"] not in drop:
                    lat[u["step"]] += u.get("latency_s") or 0
            seq.append(sum(lat.values()))
            par.append(max(sum(lat[st] for st in steps) for steps in CHAINS.values()))
    for conc in (4, 8):
        rows.append({"경우": name, "질의 동시": conc,
                     "사슬 차례로 분": np.mean(seq) * N / conc / 60, "사슬 동시 분": np.mean(par) * N / conc / 60})
pd.DataFrame(rows).round(1)

,경우,질의 동시,사슬 차례로 분,사슬 동시 분
0,검색 바뀜(모든 단계 새로),4,80.7,44.6
1,검색 바뀜(모든 단계 새로),8,40.4,22.3
2,생성만 바뀜(답변 무관 단계 저장 응답),4,34.0,26.7
3,생성만 바뀜(답변 무관 단계 저장 응답),8,17.0,13.3


## 동시 생성 영향

반복 0은 차례로(동시 1), 반복 1·2는 동시 4로 같은 날 생성했다. 이전 S14 v1·v2(dev-small 20질의, 차례로)와도 비교한다.

In [13]:
def gen_times(rows):
    first = [r["first_token_s"] for r in rows if r["first_token_s"] is not None]
    total = [r["total_s"] for r in rows]
    tout = [(r["usage"] or {}).get("completion_tokens") or 0 for r in rows]
    return {"질의": len(rows), "동시 수": dict(Counter(r.get("concurrency") for r in rows)),
            "첫 토큰 중앙값": np.median(first), "첫 토큰 p95": np.percentile(first, 95),
            "전체 중앙값": np.median(total), "전체 p95": np.percentile(total, 95),
            "출력 토큰 중앙값": np.median(tout),
            "검색 중앙값": np.median([r["retrieve_s"] for r in rows]) if "retrieve_s" in rows[0] else np.nan}


rows = {f"S24 반복 {g}": gen_times(gens[g]) for g in GEN_REPS}
rows |= {k: gen_times(read(p)) for k, p in S14_RUNS.items()}
pd.DataFrame(rows).T

,질의,동시 수,첫 토큰 중앙값,첫 토큰 p95,전체 중앙값,전체 p95,출력 토큰 중앙값,검색 중앙값
S24 반복 0,100,{1: 100},0.601079,1.642335,1.538584,3.49608,76.0,0.101973
S24 반복 1,100,{4: 100},0.594366,1.272433,1.652245,3.85038,79.5,0.104829
S24 반복 2,100,{4: 100},0.59872,0.963491,1.498272,3.348661,79.5,0.088799
S14 v1,20,{None: 20},0.559929,0.734628,1.579512,4.507239,81.5,NaN
S14 v2,20,{None: 20},0.835767,2.201099,2.517173,10.037865,93.0,NaN


## 관찰

- 잡음: 같은 답변을 judge만 다시 채점한 질의 단위 표준편차는 충실성 0.073, 관련성 0.031, 포함도 0.060, 거부 정확도(참고) 0.107이다. 포함도와 거부 정확도는 생성 반복 사이 차이가 judge 잡음보다 작게 나왔는데, 핵심 진술(`key_statements`)·근거 청크 판정·항목 나누기(`items`)의 요청이 답변과 무관해 생성 반복끼리 저장 응답을 같이 쓰기 때문이다. 그래서 n은 생성·judge 반복이 모두 다른 짝으로 정한다
- n: 생성·judge 모두 다시의 반폭 h(100)은 충실성 0.053, 관련성 0.035, 포함도 0.047, 거부 정확도 0.065다. δ = 0.03이면 필요한 n은 충실성 324, 관련성 138, 포함도 268, 거부 정확도 513, δ = 0.05면 117, 50, 97, 185다. h(n)은 n0 = 100의 표준편차가 n이 커져도 같다는 가정으로 늘린 값이다
- 실행 설정: `nothink`은 반복 사이 일관성이 높지만(충실성 |차| 0.010, `mixed` 0.087), 같은 답변·같은 진술(98/100질의에서 진술 목록 동일)에서 `verdicts`의 0 판정이 2.6%로 `mixed`(20.5\~22.5%)보다 훨씬 적고 충실성 평균이 0.975다. 일관성의 상당 부분은 거의 언제나 1로 판정하는 데서 나온다. S14 검수에서 사고 과정을 끈 `verdicts`가 확인되지 않은 질의자 주장 규칙을 적용하지 못해 low로 켰고(D-40), 판단 모델 파일럿(34_01)에서도 관대한 판정이 사람이 낮게 준 질의를 놓쳤다. 일관성만으로 고르면 관대한 설정이 뽑힌다
- `max_tokens` 16384: `mixed` 새 호출 2,943회 중 끝나지 않은 응답은 1건이고 내용 필터(`content_filter`)였다. 길이로 잘린 응답은 0건이다
- 비용: 새 호출 기준 비피크 `mixed` $2.03, `nothink` $0.56, 생성 약 $0.07(300건). 승인 추정(비피크 $4.6) 안이다
- 시간: `mixed` 100질의 채점은 질의 동시 4로 약 24.5분이었다. 질의 안의 독립 사슬을 동시에 부르면(측정 뒤 구현, 1ff3091) 같은 호출 지연으로 약 13.5분으로 줄어든다(어림)
- 동시 생성: 첫 토큰 중앙값은 동시 1(0.60초)과 동시 4(0.59·0.60초)가 같고, p95는 동시 4가 더 작았다(1.27·0.96초 대 1.64초). 검색 근거는 300건 모두 같다. 동시 생성이 지연을 늘린 흔적은 없다. S14 v2(첫 토큰 중앙값 0.84초)는 다른 날 측정이라 직접 비교하지 않는다

## 결정

사용자 결정(2026-10-26, D-50): 판정용 질의 수 n = 330, judge 실행 설정은 `deepseek_go_mixed` 유지(다수결 없음, `max_tokens` 16384). `verdicts`만 low인 설정은 위 비교로 보고 채택하지 않았다. LangSmith 추적은 대량 실행에서 끈다. judge 질의 동시 수 8은 생성 베이스라인 첫 실행에서 지연과 오류를 재고 문제가 없으면 8로 한다.